### Load and inspect the trap metadata table

In [ ]:
from pathlib import Path
import pandas as pd

In [4]:
metadata_path = Path("../data/interim/trap_metadata.csv")

print("Does path to metadata exists:", metadata_path.exists())
print("Full path to metadata:", metadata_path.resolve())

Does path to metadata exists: True
Full path to metadata: /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/interim/trap_metadata.csv


In [8]:
metadata_clean_df = pd.read_csv(metadata_path)

print("Shape of metadata table:", metadata_clean_df.shape)
print("\nColumns of metadata table:", metadata_clean_df.columns.tolist())
metadata_clean_df.head()

Shape of metadata table: (31, 7)

Columns of metadata table: ['site_id', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']


,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-02
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-02
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-02
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-02
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-02


### Select one known site and build its trap-specific chart URL

In [9]:
site_id = 29224

chart_url = (
    "https://maps.eddmaps.org/line/sitedatabyyear/"
    f"?aggregate=sum&sub=10378&project=1441&proj=1441&site={site_id}&showdate"
)

print(chart_url)

https://maps.eddmaps.org/line/sitedatabyyear/?aggregate=sum&sub=10378&project=1441&proj=1441&site=29224&showdate


### Request the chart HTML with browser-like headers

In [10]:
import requests

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    )
}

response = requests.get(chart_url, headers=headers)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

Status code: 200
Content type: text/html;charset=UTF-8
HTML length: 16930


In [12]:
print(response.text[:10000])


<!DOCTYPE html>
<html lang="en">
<head>
    <script  src="/addins/jquery/jquery-1.11.0.min.js"></script>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Trap data for silverleaf whitefly (Bemisia argentifolii) at Trap 104 using observation date</title>
    <script src="https://bugwoodcloud.org/CDN/echarts/5.5.0/echarts.min.js"></script>
    <script src="https://bugwoodcloud.org/CDN/moment.js/2.6.0/moment.min.js"></script>
<script async src="https://www.googletagmanager.com/gtag/js?id=G-C0NYEQK0GF"></script>
                            <script>
                            window.dataLayer = window.dataLayer || [];
                            function gtag(){dataLayer.push(arguments);}
                            gtag('js', new Date());

                            gtag('config', 'G-C0NYEQK0GF');
                            </script></head>
<body>
    
    <l

That confirms the chart page is only the container. The weekly counts are fetched separately from data.cfm.

### Convert the relative AJAX URL into a full URL and then Request it

In [13]:
from urllib.parse import urljoin

ajax_relative_url = (
    "data.cfm?_=752590.711879&aggregate=sum&sub=10378&end=2026"
    "&proj=1441&states=&UseDate=actual&siteid=29224"
)

data_url = urljoin(chart_url, ajax_relative_url)

print(data_url)

https://maps.eddmaps.org/line/sitedatabyyear/data.cfm?_=752590.711879&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid=29224


In [14]:
data_response = requests.get(data_url, headers=headers)

print("Status code:", data_response.status_code)
print("Content-Type:", data_response.headers.get("Content-Type"))
print("Response length:", len(data_response.text))
print("First 500 characters:")
print(data_response.text[:500])

Status code: 200
Content-Type: text/html;charset=UTF-8
Response length: 18340
First 500 characters:

[

	
	{
	"id" : "2020",
	"sub" : "2020",
	"values" : [
	{
				"value" : 0,
				"label" : "2024-04-22"
			}
			,{
				"value" : 0,
				"label" : "2024-04-28"
			}
			,{
				"value" : 0,
				"label" : "2024-05-06"
			}
			,{
				"value" : 0,
				"label" : "2024-05-15"
			}
			,{
				"value" : 0,
				"label" : "2024-05-22"
			}
			,{
				"value" : 0,
				"label" : "2024-05-28"
			}
			,{
				"value" : 0,
				"label" : "2024-06-12"
			}
			,{
				"value" : 0,
	


### Parse the response with .json() and inspect the structure

In [15]:
data = data_response.json()

print("Python object type:", type(data))
print("Number of year-series objects:", len(data))

first_year = data[0]

print("First year object type:", type(first_year))
print("Keys in first year object:", first_year.keys())
print("First year id:", first_year["id"])
print("Number of values in the first year:", len(first_year["values"]))

print("First five points:")
first_year["values"][:5]

Python object type: <class 'list'>
Number of year-series objects: 7
First year object type: <class 'dict'>
Keys in first year object: dict_keys(['id', 'sub', 'values'])
First year id: 2020
Number of values in the first year: 30
First five points:


[{'value': 0, 'label': '2024-04-22'},
 {'value': 0, 'label': '2024-04-28'},
 {'value': 0, 'label': '2024-05-06'},
 {'value': 0, 'label': '2024-05-15'},
 {'value': 0, 'label': '2024-05-22'}]

### Flatten only the first year first

In [32]:
site_id = 29224

plotted_date=[]
date_collected=[]
count=[]
year=[]

for series in data:
    series_year = series["id"]

    for point in series["values"]:
        date_plotted = point["label"]
        plotted_date.append(date_plotted)

        collection_date = series_year + "-" + date_plotted[5:]
        date_collected.append(collection_date)

        weekly_count = point["value"]
        count.append(weekly_count)

        year.append(series_year)

df = pd.DataFrame(
    {
        "site_id": site_id,
        "year": year,
        "date_collected": date_collected,
        "plotted_date": plotted_date,
        "whitefly_count": count
    }
)

df.head()

,site_id,year,date_collected,plotted_date,whitefly_count
0,29224,2020,2020-04-22,2024-04-22,0
1,29224,2020,2020-04-28,2024-04-28,0
2,29224,2020,2020-05-06,2024-05-06,0
3,29224,2020,2020-05-15,2024-05-15,0
4,29224,2020,2020-05-22,2024-05-22,0


In [33]:
print("Shape:", df.shape)

print("\nRows by year:")
print(df.groupby("year").size())

print("\nWhitefly count summary:")
print(df["whitefly_count"].describe())

Shape: (301, 5)

Rows by year:
year
2020    30
2021    50
2022    49
2023    50
2024    49
2025    51
2026    22
dtype: int64

Whitefly count summary:
count     301.000000
mean       33.627907
std       148.763933
min         0.000000
25%         0.000000
50%         0.000000
75%         3.000000
max      2000.000000
Name: whitefly_count, dtype: float64


### Rediscover the AJAX URL programmatically

Right now, we are copying the AJAX URL from the HTML. But for a reusable workflow, we should be able to find that relative data.cfm URL from the chart HTML automatically.

We can use regular expression to extract the URL.

In [34]:
import re
from urllib.parse import urljoin

In [37]:
pattern = r'url:\s*"(data\.cfm[^"]+)"'

match = re.search(pattern, response.text)

print("Found AJAX URL:", match is not None)
print(match)

if match:
    ajax_relative_url = match.group(1)
    data_url = urljoin(chart_url, ajax_relative_url)

    print("\nRelative AJAX URL:")
    print(ajax_relative_url)

    print("\nFull data URL:")
    print(data_url)

Found AJAX URL: True
<re.Match object; span=(7079, 7189), match='url: "data.cfm?_=752590.711879&aggregate=sum&sub=>

Relative AJAX URL:
data.cfm?_=752590.711879&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid=29224

Full data URL:
https://maps.eddmaps.org/line/sitedatabyyear/data.cfm?_=752590.711879&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid=29224


### Turn the logic into a small reusable function

In [39]:
def extract_weekly_count_data(site_id, headers):

    chart_url = (
    "https://maps.eddmaps.org/line/sitedatabyyear/"
    f"?aggregate=sum&sub=10378&project=1441&proj=1441&site={site_id}&showdate"
    )

    response = requests.get(chart_url, headers=headers)
    response.raise_for_status()

    pattern = r'url:\s*"(data\.cfm[^"]+)"'
    match = re.search(pattern, response.text)

    if match is None:
        raise ValueError(f"Could not find AJAX data URL for site_id={site_id}")

    ajax_relative_url = match.group(1)
    data_url = urljoin(chart_url, ajax_relative_url)

    data_response = requests.get(data_url, headers=headers)
    data_response.raise_for_status()

    data_response_json = data_response.json()

    year = []
    date_plotting = []
    date_collection = []
    whitefly_count = []
    for series in data_response_json:
        series_year = series["id"]

        for point in series["values"]:
            plot_date = point["label"]
            date_plotting.append(plot_date)

            collection_date = series_year + "-" + plot_date[5:]
            date_collection.append(collection_date)

            count = point["value"]
            whitefly_count.append(count)

            year.append(series_year)

    df = pd.DataFrame(
        {
            "site_id": site_id,
            "year": year,
            "plotted_date": date_plotting,
            "date_collected": date_collection,
            "whitefly_count": whitefly_count
        }
    )

    return df

### Test the function on Trap 104 / site 29224

In [42]:
trap_104_df = extract_weekly_count_data(29224, headers)

print("Shape:", trap_104_df.shape)

print("\nRows by year:")
print(trap_104_df.groupby("year").size())

print("\nFirst 5 rows:")
display(trap_104_df.head())

print("\nLast 5 rows:")
display(trap_104_df.tail())

Shape: (301, 5)

Rows by year:
year
2020    30
2021    50
2022    49
2023    50
2024    49
2025    51
2026    22
dtype: int64

First 5 rows:


,site_id,year,plotted_date,date_collected,whitefly_count
0,29224,2020,2024-04-22,2020-04-22,0
1,29224,2020,2024-04-28,2020-04-28,0
2,29224,2020,2024-05-06,2020-05-06,0
3,29224,2020,2024-05-15,2020-05-15,0
4,29224,2020,2024-05-22,2020-05-22,0



Last 5 rows:


,site_id,year,plotted_date,date_collected,whitefly_count
296,29224,2026,2024-05-04,2026-05-04,0
297,29224,2026,2024-05-11,2026-05-11,0
298,29224,2026,2024-05-18,2026-05-18,0
299,29224,2026,2024-05-26,2026-05-26,0
300,29224,2026,2024-06-02,2026-06-02,0


### Test the function on a second known site

In [43]:
trap_105_df = extract_weekly_count_data(29225, headers)

print("Shape:", trap_105_df.shape)

print("\nRows by year:")
print(trap_105_df.groupby("year").size())

print("\nFirst 5 rows:")
display(trap_105_df.head())

print("\nLast 5 rows:")
display(trap_105_df.tail())

Shape: (290, 5)

Rows by year:
year
2020    30
2021    47
2022    48
2023    47
2024    50
2025    47
2026    21
dtype: int64

First 5 rows:


,site_id,year,plotted_date,date_collected,whitefly_count
0,29225,2020,2024-04-22,2020-04-22,0
1,29225,2020,2024-04-28,2020-04-28,0
2,29225,2020,2024-05-06,2020-05-06,0
3,29225,2020,2024-05-15,2020-05-15,0
4,29225,2020,2024-05-22,2020-05-22,0



Last 5 rows:


,site_id,year,plotted_date,date_collected,whitefly_count
285,29225,2026,2024-05-04,2026-05-04,0
286,29225,2026,2024-05-11,2026-05-11,0
287,29225,2026,2024-05-18,2026-05-18,0
288,29225,2026,2024-05-26,2026-05-26,0
289,29225,2026,2024-06-02,2026-06-02,0


### Test the function on a tiny subset of metadata sites

In [61]:
test_sites_ids = metadata_clean_df["site_id"].tolist()
test_sites_ids[:5]

[29221, 29222, 29224, 29225, 29226]

In [63]:
small_df = []
success_log = []
failure_log = []
for site in test_sites_ids:
    try:
        df = extract_weekly_count_data(site, headers)
        small_df.append(df)

        success_log.append(
            {
                "site_id": site,
                "size": len(df),
                "first_date": df["date_collected"][0],
                "last_date": df["date_collected"][len(df)-1]
            }
        )

        print(f"Success: site_id={site}, rows={len(df)}")

    except Exception as e:
        failure_log.append(
            {
                "site_id": site,
                "error": str(e)
            }
        )

        print(f"Failed: site_id={site}, error={str(e)}")

df_concatenated = pd.concat(small_df, ignore_index=True)

print("\nCombined shape:", df_concatenated.shape)
print("\nSuccess log:")
display(pd.DataFrame(success_log).head())

print("\nFailure log:")
display(pd.DataFrame(failure_log).head())

Success: site_id=29221, rows=298
Success: site_id=29222, rows=288
Success: site_id=29224, rows=301
Success: site_id=29225, rows=290
Success: site_id=29226, rows=297
Success: site_id=29227, rows=297
Success: site_id=29228, rows=297
Success: site_id=29229, rows=295
Success: site_id=29230, rows=297
Success: site_id=29231, rows=297
Success: site_id=29232, rows=294
Success: site_id=29233, rows=299
Success: site_id=29234, rows=296
Success: site_id=29235, rows=295
Success: site_id=29236, rows=293
Success: site_id=29237, rows=301
Success: site_id=29238, rows=297
Success: site_id=29239, rows=288
Success: site_id=29240, rows=299
Success: site_id=31136, rows=49
Success: site_id=31139, rows=49
Success: site_id=31141, rows=49
Success: site_id=31142, rows=49
Success: site_id=31144, rows=49
Success: site_id=31147, rows=49
Success: site_id=31565, rows=22
Success: site_id=31566, rows=22
Success: site_id=31627, rows=53
Success: site_id=32085, rows=32
Success: site_id=32409, rows=8
Success: site_id=32410

,site_id,size,first_date,last_date
0,29221,298,2020-04-22,2026-06-02
1,29222,288,2020-04-22,2026-06-02
2,29224,301,2020-04-22,2026-06-02
3,29225,290,2020-04-22,2026-06-02
4,29226,297,2020-04-22,2026-06-02



Failure log:


""


### Validate the full combined weekly table

In [64]:
print("Shape:", df_concatenated.shape)

print("\nNumber of unique site IDs:")
print(df_concatenated["site_id"].nunique())

print("\nNumber of rows by site:")
print(df_concatenated.groupby("site_id").size().sort_values(ascending=False))

print("\nNumber of rows by year:")
print(df_concatenated.groupby("year").size().sort_values(ascending=False))

print("\nMissing values:")
print(df_concatenated.isna().sum())

print("\nWhitefly count summary:")
print(df_concatenated["whitefly_count"].describe())

Shape: (6058, 5)

Number of unique site IDs:
31

Number of rows by site:
site_id
29237    301
29224    301
29233    299
29240    299
29221    298
29227    297
29238    297
29230    297
29231    297
29226    297
29228    297
29234    296
29229    295
29235    295
29232    294
29236    293
29225    290
29239    288
29222    288
31627     53
31136     49
31139     49
31141     49
31142     49
31144     49
31147     49
32085     32
31565     22
31566     22
32409      8
32410      8
dtype: int64

Number of rows by year:
year
2025    1270
2021     940
2024     938
2023     935
2022     912
2020     566
2026     497
dtype: int64

Missing values:
site_id           0
year              0
plotted_date      0
date_collected    0
whitefly_count    0
dtype: int64

Whitefly count summary:
count    6058.000000
mean       30.459723
std       133.091348
min         0.000000
25%         0.000000
50%         0.000000
75%         7.000000
max      3300.000000
Name: whitefly_count, dtype: float64


### Convert date columns

In [65]:
df_concatenated["date_collected"] = pd.to_datetime(df_concatenated["date_collected"])
df_concatenated["plotted_date"] = pd.to_datetime(df_concatenated["plotted_date"])

print("Column types:")
print(df_concatenated.dtypes)

Column types:
site_id                    int64
year                         str
plotted_date      datetime64[us]
date_collected    datetime64[us]
whitefly_count             int64
dtype: object


In [66]:
df_concatenated.head()

,site_id,year,plotted_date,date_collected,whitefly_count
0,29221,2020,2024-04-22,2020-04-22,0
1,29221,2020,2024-04-28,2020-04-28,0
2,29221,2020,2024-05-06,2020-05-06,0
3,29221,2020,2024-05-15,2020-05-15,0
4,29221,2020,2024-05-22,2020-05-22,0


### Merge weekly counts with trap metadata

In [69]:
trap_week_df = df_concatenated.merge(
    metadata_clean_df,
    on="site_id",
    how="left"
)

print("Merged trap-week shape:", trap_week_df.shape)
print("Weekly count shape:", df_concatenated.shape)
print("Metadata shape:", metadata_clean_df.shape)

print("\nColumns:")
print(trap_week_df.columns.tolist())

print("\nMissing values after merge:")
print(trap_week_df.isna().sum())

print("\nRows by status:")
print(trap_week_df["status"].value_counts(dropna=False))

print("\nFirst 5 rows:")
display(trap_week_df.head())

Merged trap-week shape: (6058, 11)
Weekly count shape: (6058, 5)
Metadata shape: (31, 7)

Columns:
['site_id', 'year', 'plotted_date', 'date_collected', 'whitefly_count', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']

Missing values after merge:
site_id            0
year               0
plotted_date       0
date_collected     0
whitefly_count     0
trap_label         0
latitude           0
longitude          0
status             0
first_report       0
last_report       60
dtype: int64

Rows by status:
status
Active     5384
OldData     674
Name: count, dtype: int64

First 5 rows:


,site_id,year,plotted_date,date_collected,whitefly_count,trap_label,latitude,longitude,status,first_report,last_report
0,29221,2020,2024-04-22,2020-04-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-02
1,29221,2020,2024-04-28,2020-04-28,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-02
2,29221,2020,2024-05-06,2020-05-06,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-02
3,29221,2020,2024-05-15,2020-05-15,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-02
4,29221,2020,2024-05-22,2020-05-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-02


### Save final merged datasets as csv files

In [73]:
from pathlib import Path

interim_path = Path("../data/interim/whitefly_weekly_counts.csv")
processed_path = Path("../data/processed/whitefly_trap_weekly_counts_with_metadata.csv")

# Make sure folders exist
interim_path.parent.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)

# Save datasets
df_concatenated.to_csv(interim_path, index=False)
trap_week_df.to_csv(processed_path, index=False)

print("Saved weekly counts to:")
print(interim_path.resolve())

print("\nSvaed merged trap-week dataset to:")
print(processed_path.resolve())

Saved weekly counts to:
/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/interim/whitefly_weekly_counts.csv

Svaed merged trap-week dataset to:
/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/processed/whitefly_trap_weekly_counts_with_metadata.csv
